## 1. Imports

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


import xgboost as xgb
# import lightgbm as lgb

## 2. Load data

In [ ]:
df = pd.read_csv("train.csv")

TARGET = "target"

print("Shape:", df.shape)
print(df.head())
print(df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nTarget distribution:\n", df[TARGET].describe())

## 3. Split features / target, then train/validation split

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# X = X.drop(columns=["id"])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 4. Identify column types
Auto-detected — check this looks right!

In [ ]:
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)
# >>> CHANGE: move any columns between the two lists above if auto-detect is wrong

## 5. Preprocessing pipeline

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
])

## 6. Choose a model

In [ ]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# >>> "baseline", "ridge", "forest", "xgb", or "lgbm"
MODEL_CHOICE = "xgb"

models = {
    "baseline": LinearRegression(),
    "ridge": Ridge(alpha=1.0),
    "forest": RandomForestRegressor(n_estimators=300, random_state=42),
    "xgb": XGBRegressor(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=5,
        random_state=42,
    ),
    "lgbm": LGBMRegressor(
        n_estimators=300,
        learning_rate=0.1,
        random_state=42,
    ),
}
model = models[MODEL_CHOICE]

reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model),
])


## 7. Cross-validation
More trustworthy than a single split.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    reg_pipeline, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error"
)
print(f"Cross-val RMSE: {-cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

## 8. Fit on training set, evaluate on held-out validation set

In [ ]:
reg_pipeline.fit(X_train, y_train)
val_preds = reg_pipeline.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, val_preds))
mae = mean_absolute_error(y_val, val_preds)
r2 = r2_score(y_val, val_preds)

print("Validation RMSE:", rmse)
print("Validation MAE:", mae)
print("Validation R2:", r2)

## 9. Automatic hyperparameter optimization\nUses `RandomizedSearchCV` with a parameter space chosen automatically for whichever model you picked in Step 6 — no manual grid-building needed. Adjust `N_ITER` if you want to trade thoroughness for speed.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist_by_model = {
    "ridge": {
        "model__alpha": uniform(0.01, 50),
    },
    "forest": {
        "model__n_estimators": randint(200, 600),
        "model__max_depth": randint(3, 20),
        "model__min_samples_split": randint(2, 10),
    },
    "xgb": {
        "model__n_estimators": randint(200, 600),
        "model__max_depth": randint(3, 8),
        "model__learning_rate": uniform(0.03, 0.17),   # 0.03 to 0.20
        "model__subsample": uniform(0.7, 0.3),          # 0.7 to 1.0
        "model__colsample_bytree": uniform(0.7, 0.3),
    },
    "lgbm": {
        "model__n_estimators": randint(200, 600),
        "model__max_depth": randint(3, 12),
        "model__learning_rate": uniform(0.03, 0.17),
        "model__subsample": uniform(0.7, 0.3),
        "model__colsample_bytree": uniform(0.7, 0.3),
        "model__num_leaves": randint(15, 100),
    },
    # "baseline" (plain LinearRegression) has no useful hyperparameters to
    # tune, so it's deliberately left out — tuning will be skipped for it.
}

N_ITER = 20

if MODEL_CHOICE in param_dist_by_model:
    search = RandomizedSearchCV(
        reg_pipeline,
        param_distributions=param_dist_by_model[MODEL_CHOICE],
        n_iter=N_ITER,
        cv=3,                     # fewer folds than Step 7 to keep this fast
        scoring="neg_root_mean_squared_error",
        random_state=42,
        n_jobs=-1,
        verbose=1,
    )
    search.fit(X_train, y_train)
    print("Best params found:", search.best_params_)
    print("Best CV RMSE:", -search.best_score_)

    reg_pipeline = search.best_estimator_
else:
    print(f"No tuning grid defined for '{MODEL_CHOICE}' — skipping tuning, keeping current pipeline.")


## 10. Retrain on all data, then predict on the real test set

In [ ]:
reg_pipeline.fit(X, y)

# >>> CHANGE: load the real test/submission file they give you
# test_df = pd.read_csv("test.csv")
# test_preds = reg_pipeline.predict(test_df)
# submission = pd.DataFrame({"id": test_df["id"], TARGET: test_preds})
# submission.to_csv("submission.csv", index=False)
# print("Saved submission.csv")

## Notes for the viva
Fill these in as you go, in your own words:
- What does the data represent, and what are you predicting?
- Why did you choose this imputation strategy for missing values?
- Why this model? What did cross-validation tell you?
- What would you try with more time?